In [1]:
import pandas as pd

# Load raw datasets
df_3w = pd.read_csv(r"E:\Research_Paper\oil_gas_ai\data\raw\cleaned_3w_dataset.csv")
df_pm = pd.read_csv(r"E:\Research_Paper\oil_gas_ai\data\raw\cleaned_predictive_maintenance.csv")
df_oreda = pd.read_csv(r"E:\Research_Paper\oil_gas_ai\data\raw\cleaned_oreda_reliability.csv")

print("3W shape:", df_3w.shape)
print("PM shape:", df_pm.shape)
print("OREDA shape:", df_oreda.shape)


3W shape: (2600000, 37)
PM shape: (124494, 12)
OREDA shape: (2600000, 15)


In [2]:
# 3W dataset has NO time column
# Treat each row as a sensor snapshot
print("3W columns:", df_3w.columns.tolist())


3W columns: ['well_id', 'pressure_gas_lift', 'pressure_production_casing', 'valvestatus_dhsv', 'valvestatus_master1', 'valvestatus_master2', 'valvestatus_xmastree', 'valvestatus_shutdowngaslift', 'valvestatus_shutdownproduction', 'valvestatus_wing1', 'valvestatus_wing2', 'valvestatus_crossover', 'pressure_annulus', 'pressure_checkgaslift', 'pressure_monitorproduction', 'pressure_downholegauge', 'pressure_toptubing', 'flow_gaslift', 'temp_checkproduction', 'temp_downholegauge', 'temp_toptubing', 'well_operatingclass', 'well_statecode', 'category', 'region', 'country', 'state', 'facility_name', 'facility_status', 'operator', 'latitude', 'longitude', 'attribute_score', 'data_source_score', 'update_frequency_score', 'aggregate_quality_score', 'well_age_years']


In [3]:
print("PM columns:", df_pm.columns.tolist())


PM columns: ['date', 'device', 'failure', 'metric1', 'metric2', 'metric3', 'metric4', 'metric5', 'metric6', 'metric7', 'metric8', 'metric9']


In [4]:
print(" columns:", df_oreda.columns.tolist())

 columns: ['equipment_id', 'asset_type', 'region', 'observation_years', 'mtbf_hours', 'mttr_hours', 'failure_rate_per_year_expected', 'failures_observed_total', 'downtime_hours_per_year', 'avg_repair_cost_usd', 'failure_rate_pct', 'dominant_failure_mode', 'availability_pct', 'mtbf_days', 'mttr_days']


In [5]:
df_pm["failure"] = df_pm["failure"].astype(int)

# Generate synthetic mapping (benchmark alignment)
df_pm["well_id"] = (
    df_pm["device"]
    .astype("category")
    .cat.codes
    .astype(str)
)

df_pm = df_pm[["well_id", "failure"]]



In [6]:
df = pd.concat([df_3w, df_pm], axis=1)


In [7]:
df_oreda["well_id"] = (
    df_oreda["equipment_id"]
    .astype("category")
    .cat.codes
    .astype(str)
)


In [8]:
df = pd.concat([df, df_oreda], axis=1)


In [9]:
leakage_cols = [
    "well_id",
    "facility_name",
    "operator",
    "latitude",
    "longitude"
]

df = df.drop(columns=[c for c in leakage_cols if c in df.columns])


In [10]:
# num_cols = df.select_dtypes(include="number").columns
# df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# cat_cols = df.select_dtypes(include="object").columns
# df[cat_cols] = df[cat_cols].fillna("Unknown")


In [11]:
df.to_csv("../data/processed/failure_dataset.csv", index=False)

print("✅ failure_dataset.csv created")
print("Failure distribution:")
print(df["failure"].value_counts(normalize=True))


✅ failure_dataset.csv created
Failure distribution:
failure
0.0    0.999149
1.0    0.000851
Name: proportion, dtype: float64


In [12]:
print(df.columns.tolist())

['pressure_gas_lift', 'pressure_production_casing', 'valvestatus_dhsv', 'valvestatus_master1', 'valvestatus_master2', 'valvestatus_xmastree', 'valvestatus_shutdowngaslift', 'valvestatus_shutdownproduction', 'valvestatus_wing1', 'valvestatus_wing2', 'valvestatus_crossover', 'pressure_annulus', 'pressure_checkgaslift', 'pressure_monitorproduction', 'pressure_downholegauge', 'pressure_toptubing', 'flow_gaslift', 'temp_checkproduction', 'temp_downholegauge', 'temp_toptubing', 'well_operatingclass', 'well_statecode', 'category', 'region', 'country', 'state', 'facility_status', 'attribute_score', 'data_source_score', 'update_frequency_score', 'aggregate_quality_score', 'well_age_years', 'failure', 'equipment_id', 'asset_type', 'region', 'observation_years', 'mtbf_hours', 'mttr_hours', 'failure_rate_per_year_expected', 'failures_observed_total', 'downtime_hours_per_year', 'avg_repair_cost_usd', 'failure_rate_pct', 'dominant_failure_mode', 'availability_pct', 'mtbf_days', 'mttr_days']
